In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-4",
)

In [ ]:
# SYSTEM_PROMPT = """
# You are an expert in parsing terminal session logs.

# Process the provided <system_output> starting from the very first line. You must be extremely careful to examine every single line sequentially without skipping.

# An event boundary always occurs between a user's input and the subsequent system output. Identify the absolute first occurrence of a shell prompt (e.g., demo@boxtop:~$ or similar) which signifies that the previous command has finished and a new event is starting.

# For each line, if it marks the start of a new event, you must return exactly one timestamp from that prompt line to define the end of the current group.

# Do not return any text, explanation, or multiple lines—only the single target timestamp.

# Return format:
# <timestamp>
# """
SYSTEM_PROMPT = """
You are an expert in parsing terminal session logs.

Extract **only the timestamps** where a shell prompt appears inside <system_output>.
Do not return any text, explanation, or formatting — only timestamps, one per line.

A shell prompt looks like:
demo@boxtop:~$
or any similar prompt indicating that the previous command finished.

Each timestamp you return **defines an event boundary**.
Remember: Only return timestamps. Each timestamp defines an event boundary. No extra text or explanation.
***Important note: The switch between the user input and system output should not be treated as a boundary.***

Return format:
<timestamp>
"""
xml_data = """
{"input": "<system_output timestamp=\"0.008584\">[?2004hdemo@stephost:~$ </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"0.603025\"/>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"0.603588\">\n(reverse-i-search)`': </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"0.937684\">s</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"0.938181\">s': [7ms[27mync</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"1.114177\">s</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"1.1146\">s': [7mss[27mh 172.16.0.17</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"1.291778\">h</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"1.292225\">[1@h': [7mssh[27m</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"2.617004\">\n</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"2.617465\">\n[8Pdemo@stephost:~$ ssh\n[?2004l\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"5.694771\">ssh: connect to host 172.16.0.17 port 22: No route to host\n\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"5.697578\">[?2004hdemo@stephost:~$ </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.138774\">c</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.139172\">c</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.367648\">m</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.368275\">m</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.491765\"> </user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.492239\"> </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.700208\">l</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.700707\">l</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.826627\">i</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.827056\">i</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"8.079826\">s</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.080258\">s</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"8.28657\">t</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.286967\">t</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"8.474631\">\n</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.475097\">\n[?2004l\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.48351\">implicitserver-tearoff-16: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.523754\">\nimplicitserver-tearoff-17: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.575367\">\nwikiserver-tearoff-3: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.604153\"> Running\nqemuwordserver-tearoff-4: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.63601\">\nlampserver-tearoff-14: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.667641\">\nwikiserver-tearoff-5: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.699481\">\ndrupalserver-tearoff-13: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.730786\">\nnullhost-tearoff-11: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.76265\">\nwikiserver-tearoff-2: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.794868\">\nfaiserver-tearoff-8: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.826481\">\nfaiserver-tearoff-9: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.858307\">\nfaiserver-tearoff-1: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.888755\">\nfaiserver-tearoff-19: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.918944\">\nnullhost-tearoff-10: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.952225\"> Running\nnullhost-tearoff-7: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.983719\">\nfaiserver-tearoff-18: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.015378\">\nfaiserver-tearoff-12: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.047348\">\nnullhost-tearoff-6: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.079908\"> Running\nnullhost-tearoff-15: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.110634\">\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.110867\">[?2004hdemo@stephost:~$ </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"15.55465\"/>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"15.555045\">[?2004l\n\nexit\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
"""

In [ ]:
# === Run inference (assumes `model` and `tokenizer` are already loaded) ===
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Extract event boundaries from this XML:\n{xml_data}"}
]

# Tokenize & move to GPU
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate from model
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.2,
)

import re

# raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]
raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

# extract the assistant block using the marker tokens, if present
m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
if m:
    raw_assistant = m.group(1).strip()
else:
    # fallback to full decode if markers missing
    raw_assistant = raw_output

print("=== RAW ASSISTANT OUTPUT ===")
print(raw_assistant)

# same extraction as above
timestamps = re.findall(r'\d+\.\d+', raw_assistant)
# dedupe-preserve-order
seen = set(); ordered = []
for t in timestamps:
    if t not in seen:
        seen.add(t); ordered.append(t)

print("\n=== CLEANED EXTRACTED TIMESTAMPS (assistant-only) ===")
print("\n".join(ordered) if ordered else "(no timestamps found)")


# === Print clearly separated input and model "true" output ===
print("=== INPUT XML ===")
print(xml_data)
print("\n=== RAW MODEL OUTPUT (decoded) ===")
print(raw_output)
print("\n=== CLEANED EXTRACTED TIMESTAMPS ===")
if timestamps:
    print("\n".join(timestamps))
else:
    print("(no timestamps found)")

=== RAW ASSISTANT OUTPUT ===
0.008584  
2.617465  
5.697578  
8.475097  
9.110867

=== CLEANED EXTRACTED TIMESTAMPS (assistant-only) ===
0.008584
2.617465
5.697578
8.475097
9.110867
=== INPUT XML ===

{"input": "<system_output timestamp="0.008584">[?2004hdemo@stephost:~$ </system_output>

[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp="0.603025"/>

[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp="0.603588">
(reverse-i-search)`': </system_output>

[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp="0.937684">s</user_input>

[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp="0.938181">s': sync</system_output>

[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp="1.114177">s</user_input>

[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp="1.1146">s': ssh 172.16.0.17</system_output>

[Script Action: Processed XML chunk.]"}
{"input": "<user_inp